# Football3D Stage 4 — HMR2 pose extraction

1. Add the `dataset stage4` Kaggle Dataset as an input (**keep it private**: it holds the SMPL file, which may not be redistributed).
2. Turn **Internet on** and select a **T4 GPU** in Notebook settings.
3. Run all cells in order. The first run downloads the 2.6 GB official HMR2 model.
4. Download `pose_1131.npz` and `pose_1131_preview.mp4` from the final cell.
5. Watch the preview: the right side should stay aligned with the white attacker.

This is inference, not training. It processes the white attacker in frames 560–620. The raw tracker splits him across IDs 944 and 1131; the joined pitch track is ID 1131.
HMR2 gets the **full frame plus the tracker box**, the way the official demo does — no pre-cropping.
The body is the **average adult male**: HMR2's joint rotations on the male SMPL model with body shape 0.

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
frame_files = [p for p in INPUT.rglob('000001.jpg') if p.parent.name.lower() == 'img1']
track_files = list(INPUT.rglob('SNGS-043_football-player-detection-v9_botsort.json'))
smpl_files = list(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl'))

assert frame_files, f'Frame folder not found under {INPUT}'
assert track_files, f'Track JSON not found under {INPUT}'
assert smpl_files, f'SMPL model not found under {INPUT}'

FRAMES = frame_files[0].parent
TRACK_JSON = track_files[0]
SMPL_SOURCE = smpl_files[0]
TRACK_ID = 1131
TRACK_PARTS = ((560, 580, 944), (581, 620, 1131))
START, END = 560, 620
WORK = Path('/kaggle/working/football3d_stage4')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'pose_1131.npz'
REPO = Path('/kaggle/working/4D-Humans')

print('Frames:', FRAMES)
print('Track JSON:', TRACK_JSON)
print('SMPL model:', SMPL_SOURCE)
print('Images:', len(list(FRAMES.glob('*.jpg'))))

In [ ]:
# Install 4DHumans and aria2. This cell is safe to run again.
import shutil, subprocess, sys

# A previous interrupted clone can leave the folder present but unusable.
if not (REPO / 'hmr2' / '__init__.py').is_file():
    if REPO.exists():
        print('Incomplete 4D-Humans checkout; replacing it.')
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shubham-goel/4D-Humans.git', str(REPO)], check=True)
assert (REPO / 'hmr2' / '__init__.py').is_file(), f'HMR2 package missing from {REPO}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[all]'], check=True)
# 4D-Humans still uses timm's deprecated compatibility import.
for source in REPO.rglob('*.py'):
    text = source.read_text()
    if 'timm.models.layers' in text:
        source.write_text(text.replace('timm.models.layers', 'timm.layers'))
# Kaggle can miss the editable-install path until its kernel restarts.
sys.path.insert(0, str(REPO.resolve()))
import hmr2
print('HMR2 import:', hmr2.__file__)

if not shutil.which('aria2c'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)

print('4DHumans installed; aria2c:', shutil.which('aria2c'))

In [ ]:
# Put SMPL where HMR2 looks for it, and collect the attacker's two raw track fragments.
import cv2, json, numpy as np, sys
sys.path.insert(0, str(REPO))
assert (REPO / 'hmr2').is_dir(), f'4D-Humans is not at {REPO}: run cell 2 first (a new Kaggle session starts with an empty /kaggle/working)'

# HMR2 loads whatever file sits at SMPL_NEUTRAL.pkl. We give it the MALE model on purpose:
# cell 5 uses it to build a male body. (For accuracy numbers later, this should be the real
# neutral model, basicModel_neutral_lbs_10_207_0_v1.0.0.pkl — HMR2 was trained on that one.)
from hmr2.configs import CACHE_DIR_4DHUMANS
fallback = Path.cwd() / 'data' / 'basicModel_neutral_lbs_10_207_0_v1.0.0.pkl'
cache_model = Path(CACHE_DIR_4DHUMANS) / 'data' / 'smpl' / 'SMPL_NEUTRAL.pkl'
fallback.parent.mkdir(parents=True, exist_ok=True)
cache_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SMPL_SOURCE, fallback)
shutil.copy2(SMPL_SOURCE, cache_model)
print('SMPL (male) copied to:', cache_model)

# The tracker box, unpadded. HMR2's dataset widens it to its own 192:256 shape and makes it square,
# exactly as in the official demo. Padding it ourselves as well made the player only ~72% of the
# network input instead of ~100%, from a player who is only ~78 px tall to begin with.
track = json.loads(TRACK_JSON.read_text())
boxes = {}
for row in track['frames']:
    if START <= row['frame'] <= END:
        raw_id = next((track_id for start, end, track_id in TRACK_PARTS if start <= row['frame'] <= end), None)
        box = next((b for b in row['boxes'] if b.get('id') == raw_id), None)
        if box is not None:
            boxes[row['frame']] = np.array(box['xyxy'], dtype=np.float32)

assert len(boxes) >= 55, f'Track {TRACK_ID} has only {len(boxes)} boxes in frames {START}-{END}'
heights = [b[3] - b[1] for b in boxes.values()]
missing = sorted(set(range(START, END + 1)) - set(boxes))
print(f'Track {TRACK_ID}: {len(boxes)} frames, {min(boxes)} to {max(boxes)}, box height median {np.median(heights):.0f} px, missing {missing}')

In [ ]:
# Download the official HMR2 model. aria2 resumes partial downloads and uses parallel connections.
import sys, tarfile
sys.path.insert(0, str(REPO))
from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import DEFAULT_CHECKPOINT

cache = Path(CACHE_DIR_4DHUMANS)
archive = cache / 'hmr2_data.tar.gz'
checkpoint = Path(DEFAULT_CHECKPOINT)
url = 'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'
cache.mkdir(parents=True, exist_ok=True)

if not checkpoint.exists():
    subprocess.run([
        'aria2c', '--continue=true', '--max-connection-per-server=16', '--split=16',
        '--min-split-size=1M', '--file-allocation=none', f'--dir={cache}',
        f'--out={archive.name}', url
    ], check=True)
    print('Download complete. Extracting...')
    # The server may deliver this already decompressed despite the .tar.gz name.
    with tarfile.open(archive, 'r:*') as bundle:
        bundle.extractall(cache, filter='data')

assert checkpoint.exists(), f'Checkpoint missing: {checkpoint}'
print('HMR2 ready:', checkpoint)

In [ ]:
# Run HMR2 on the full frames (tracker box), build a male body, save pose data plus a visual check video.
import torch
import subprocess
from hmr2.models import load_hmr2, DEFAULT_CHECKPOINT
from hmr2.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD
from hmr2.utils import recursive_to
from hmr2.utils.renderer import Renderer

checkpoint = Path(DEFAULT_CHECKPOINT)
assert checkpoint.exists(), f'Checkpoint missing: {checkpoint}. Run the download cell first.'
assert torch.cuda.is_available(), 'GPU is not enabled. Select a GPU accelerator in Kaggle settings.'
device = torch.device('cuda')
print('Using:', torch.cuda.get_device_name(0))
# PyTorch 2.6+ defaults to weights_only=True. HMR2's trusted Lightning
# checkpoint contains an OmegaConf object, so load it in legacy mode.
original_torch_load = torch.load
def trusted_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_torch_load(*args, **kwargs)
torch.load = trusted_torch_load
try:
    model, model_cfg = load_hmr2(str(checkpoint))
finally:
    torch.load = original_torch_load
model = model.to(device).eval()

renderer = Renderer(model_cfg, faces=model.smpl.faces)
WORK.mkdir(parents=True, exist_ok=True)
preview_path = WORK / 'pose_1131_preview.mp4'
preview = subprocess.Popen([
    'ffmpeg', '-y', '-loglevel', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24',
    '-s', '512x256', '-r', '25', '-i', '-', '-an', '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p', str(preview_path)
], stdin=subprocess.PIPE)
frames_out, joints_out, vertices_out, cameras_out, translations_out = [], [], [], [], []
orientations_out, body_poses_out, betas_out = [], [], []
for index, frame in enumerate(sorted(boxes), 1):
    image = cv2.imread(str(FRAMES / f'{frame:06d}.jpg'))
    assert image is not None, f'Missing frame {frame}'
    dataset = ViTDetDataset(model_cfg, image, boxes[frame][None])   # full frame + tracker box, like the demo
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)
    batch = recursive_to(next(iter(loader)), device)
    with torch.inference_mode():
        result = model(batch)
        params = result['pred_smpl_params']
        # Male body on purpose. HMR2 predicts betas (body shape) for SMPL's NEUTRAL body; the same
        # 10 numbers read by the MALE model give a female-looking chest and hips. Joint rotations
        # carry over between SMPL models, so keep HMR2's rotations and use betas = 0 (average male).
        male = model.smpl(global_orient=params['global_orient'].float(),
                          body_pose=params['body_pose'].float(),
                          betas=torch.zeros_like(params['betas']).float(), pose2rot=False)
    vertices = male.vertices[0]
    joints = torch.einsum('jk,kv->jv', model.smpl.J_regressor.to(device), vertices)
    input_patch = batch['img'][0].cpu() * (DEFAULT_STD[:, None, None] / 255) + (DEFAULT_MEAN[:, None, None] / 255)
    input_patch = input_patch.permute(1, 2, 0).numpy()
    rendered = renderer(vertices.cpu().numpy(), result['pred_cam_t'][0].cpu().numpy(), batch['img'][0])
    side_by_side = np.concatenate([input_patch, rendered], axis=1)
    preview.stdin.write(np.clip(255 * side_by_side, 0, 255).astype(np.uint8).tobytes())
    frames_out.append(frame)
    joints_out.append(joints.cpu().numpy())
    vertices_out.append(vertices.cpu().numpy())
    cameras_out.append(result['pred_cam'][0].cpu().numpy())
    translations_out.append(result['pred_cam_t'][0].cpu().numpy())
    orientations_out.append(params['global_orient'][0].cpu().numpy())
    body_poses_out.append(params['body_pose'][0].cpu().numpy())
    betas_out.append(params['betas'][0].cpu().numpy())                # HMR2's own (neutral-body) shape, kept for reference only
    if index == 1 or index % 10 == 0 or index == len(boxes):
        print(f'Processed {index}/{len(boxes)}')

preview.stdin.close()
assert preview.wait() == 0 and preview_path.exists() and preview_path.stat().st_size > 0, f'Preview was not written: {preview_path}'
np.savez_compressed(
    OUT, frame=np.asarray(frames_out), joints=np.asarray(joints_out),
    vertices=np.asarray(vertices_out), pred_cam=np.asarray(cameras_out),
    pred_cam_t=np.asarray(translations_out),
    global_orient_rotmat=np.asarray(orientations_out),
    body_pose_rotmat=np.asarray(body_poses_out), betas=np.asarray(betas_out),
    fps=np.float32(25)
)
saved = np.load(OUT)
required = {'global_orient_rotmat', 'body_pose_rotmat', 'betas'}
assert required <= set(saved.files), f'Missing SMPL variables: {required - set(saved.files)}'
print('Saved:', OUT)
print('Preview:', preview_path)
print('Joints:', np.asarray(joints_out).shape, 'Vertices:', np.asarray(vertices_out).shape)
print('SMPL variables:', {name: saved[name].shape for name in sorted(required)})

In [ ]:
# Watch the preview here, then download both files.
from IPython.display import FileLink, Video
print(f'Preview size: {preview_path.stat().st_size / 1e6:.1f} MB')
display(Video(str(preview_path), embed=True))
display(FileLink(str(OUT)))
display(FileLink(str(preview_path)))